# Module 2 — Data Exploration

Pull small samples of each data layer and print the head, just to see what the raw shapes look like before designing the real pipeline.

1. Raw scientific text — arXiv cond-mat abstracts (free API, no key)
2. Structured material records — Materials Project (needs `MP_API_KEY` env var; falls back to a tiny embedded sample if not set)
3. SFT instruction examples — templated from the structured records
4. DPO preference pairs — grounded vs hallucinated, templated from the structured records

In [56]:
import os
import json
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET

DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Raw scientific text — arXiv abstracts

Query the arXiv API for `cond-mat.mtrl-sci` abstracts. No API key required.

In [57]:
def fetch_arxiv_abstracts(query="cat:cond-mat.mtrl-sci", max_results=20):
    url = "http://export.arxiv.org/api/query"
    params = {
        "search_query": query,
        "start": 0,
        "max_results": max_results,
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    ns = {"atom": "http://www.w3.org/2005/Atom"}
    root = ET.fromstring(resp.text)
    records = []
    for entry in root.findall("atom:entry", ns):
        records.append({
            "id": entry.find("atom:id", ns).text,
            "title": entry.find("atom:title", ns).text.strip().replace("\n", " "),
            "summary": entry.find("atom:summary", ns).text.strip().replace("\n", " "),
            "published": entry.find("atom:published", ns).text,
        })
    return pd.DataFrame(records)

df_text = fetch_arxiv_abstracts()
df_text.to_json(f"{DATA_DIR}/raw_text_arxiv.jsonl", orient="records", lines=True)
print(df_text.shape)
df_text.head()

(20, 4)


,id,title,summary,published
0,http://arxiv.org/abs/2606.17012v1,Evolution of Nonlinear Ion Transport in Nanopo...,Understanding the ionic transport and scaling ...,2026-06-15T17:45:09Z
1,http://arxiv.org/abs/2606.16994v1,Hydrogen Chemisorption and Current-Induced Spi...,Topological semimetals have been proposed as e...,2026-06-15T17:30:48Z
2,http://arxiv.org/abs/2606.16931v1,Dynamical Steering and Unambiguous Signature o...,Dynamical manipulation of Majorana zero modes ...,2026-06-15T16:30:34Z
3,http://arxiv.org/abs/2606.16851v1,High-entropy Fe$_2$VAl-based thermoelectric mo...,Thermoelectric (TE) materials enable the direc...,2026-06-15T15:26:23Z
4,http://arxiv.org/abs/2606.16850v1,PF-DIC: Phase field digital image correlation ...,This work presents a novel digital image corre...,2026-06-15T15:26:22Z


## 1a. Token count per summary

Quick sanity check on how long each abstract is in tokens, using `tiktoken` (GPT-style BPE tokenizer) as a stand-in -- the real tokenizer will depend on the base model chosen later.

In [58]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

df_text["summary_token_count"] = df_text["summary"].apply(lambda s: len(encoding.encode(s)))
print(df_text[["title", "summary_token_count"]].to_string(index=False))
print()
print(df_text["summary_token_count"].describe())

                                                                                                                                               title  summary_token_count
                                Evolution of Nonlinear Ion Transport in Nanopore Arrays: Ionic Conductance, Current Rectification, and Osmotic Power                  299
                                                                                 Hydrogen Chemisorption and Current-Induced Spin Polarization on NbP                  265
                                          Dynamical Steering and Unambiguous Signature of Majorana Corner Modes in Altermagnetic Josephson Junctions                  220
                                                             High-entropy Fe$_2$VAl-based thermoelectric modules with improved conversion efficiency                  351
                               PF-DIC: Phase field digital image correlation for integrated full-field displacement, strain, and damage measurements  

## 1b. Raw scientific text — full paper text

The Atom API above only returns the abstract. To see a full paper, download the PDF for one sample (the first row of `df_text`) and extract its text. Toggle `FETCH_FULL_TEXT = False` to skip this (it's slower and noisier than abstracts).

In [59]:
FETCH_FULL_TEXT = True

def fetch_full_text(arxiv_id):
    from pypdf import PdfReader
    import io

    pdf_url = f"https://arxiv.org/pdf/{arxiv_id}"
    resp = requests.get(pdf_url, timeout=30)
    resp.raise_for_status()
    reader = PdfReader(io.BytesIO(resp.content))
    return "\n".join(page.extract_text() for page in reader.pages)

if FETCH_FULL_TEXT:
    sample_id = df_text.iloc[0]["id"].rsplit("/", 1)[-1]
    full_text = fetch_full_text(sample_id)
    with open(f"{DATA_DIR}/sample_full_paper.txt", "w") as f:
        f.write(full_text)
    print(f"arxiv id: {sample_id}")
    print(f"characters: {len(full_text)}")
    print()
    print(full_text[:-20000])

arxiv id: 2606.17012v1
characters: 70524

1 
 
 
Evolution of Nonlinear Ion Transport in Nanopore Arrays: 
Ionic Conductance, Current Rectification, and Osmotic 
Power  
 
 
Chih-Yuan Lin1 and Marija Drndić1* 
 
1Department of Physics and Astronomy, University of Pennsylvania, Philadelphia, PA 19104, United 
States 
 
 
*email: drndic@physics.upenn.edu 
 
 
 
 
KEYWORD 
Nanopore array, ion transport, scaling conductance, current rectification, osmotic 
energy  
  
2 
 
ABSTRACT 
Understanding the ionic transport and scaling behaviors in nanopore arrays is essential 
for bridging fundamental ion physics and blue energy applications.  By fabricating sub-3 
nm and sub- 20 nm diameter nanopore arrays (NPAs)  spanning from few pores to N ~ 
10000, we systematically investigate ionic conductance, ion current rectification, and 
osmotic energy conversion. We report the ionic conductance scal ing laws and 
nonlinearity with nanopore number, with stronger deviations from linearity at lower salt

## 1c. Cleaning the full paper text

Instead of guessing word breaks line by line, split the paper into sections by detecting
header-shaped lines (short, ALL CAPS, e.g. `ABSTRACT`, `INTRODUCTION`, `RESULTS AND DISCUSSION`)
without hardcoding which section names to expect. Sections are kept as a plain numbered list
(not a dict), then page numbers and stray whitespace are cleaned up within each section.

In [60]:
import re

def is_section_header(line):
    stripped = line.strip()
    letters = [ch for ch in stripped if ch.isalpha()]
    # a header-shaped line: short, mostly letters, and fully uppercase -- no assumption about which words
    return bool(letters) and len(stripped) <= 60 and stripped == stripped.upper()

def split_into_sections(text):
    lines = text.split("\n")
    sections = []
    current = []
    for line in lines:
        if is_section_header(line) and current:
            sections.append("\n".join(current))
            current = []
        current.append(line)
    if current:
        sections.append("\n".join(current))
    return sections

def clean_section(text):
    # drop standalone page-number lines, e.g. "12"
    text = re.sub(r"\n\s*\d{1,4}\s*\n", "\n", text)
    # collapse repeated blank lines and excess spaces
    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()

if FETCH_FULL_TEXT:
    raw_sections = split_into_sections(full_text)
    cleaned_sections = [clean_section(s) for s in raw_sections]
    cleaned_text = "\n\n".join(cleaned_sections)

    with open(f"{DATA_DIR}/sample_full_paper_clean.txt", "w") as f:
        f.write(cleaned_text)

    # save just the preview snippets to separate files so before/after is easy to diff
    PREVIEW_CHARS = 1000
    with open(f"{DATA_DIR}/before_snippet.txt", "w") as f:
        f.write(full_text[:PREVIEW_CHARS])
    with open(f"{DATA_DIR}/after_snippet.txt", "w") as f:
        f.write(cleaned_text[:PREVIEW_CHARS])

    print(f"detected {len(raw_sections)} sections:")
    for i, s in enumerate(cleaned_sections, 1):
        first_line = s.strip().split("\n")[0][:60]
        print(f"  {i}. {first_line}")
    print()
    print(f"raw characters: {len(full_text)} -> cleaned characters: {len(cleaned_text)}")
    print(f"preview snippets saved to {DATA_DIR}/before_snippet.txt and {DATA_DIR}/after_snippet.txt")

detected 19 sections:
  1. 1 
  2. KEYWORD 
  3. ABSTRACT 
  4. INTRODUCTION 
  5. RESULTS AND DISCUSSION 
  6. CONCLUSIONS 
  7. MATERIALS AND METHODS 
  8. C
  9. EBL/RIE 
  10. EBL/RIE 
  11. ASSOCIATED CONTENT 
  12. NOTES 
  13. ACKNOWLEDGMENTS 
  14. REFERENCES 
  15. 23 (38), 385308. DOI: 10.1088/0957-4484/23/38/385308. 
  16. 2015, 26 (31). DOI: 10.1088/0957-4484/26/31/314001. 
  17. 26 (1), 012005. DOI: 10.1063/1.4863206. 
  18. DOI: 10.1063/5.0044227. 
  19. 3065. DOI: 10.1116/1.2366698. 

raw characters: 70524 -> cleaned characters: 69945
preview snippets saved to ../data/before_snippet.txt and ../data/after_snippet.txt


## 2. Structured material records — Materials Project

Requires a free API key from https://materialsproject.org/api (set `MP_API_KEY` in your environment). If no key is set, falls back to a small embedded sample so the rest of the notebook still runs.

In [61]:
MP_API_KEY = os.environ.get("MP_API_KEY")

FALLBACK_RECORDS = [
    {"material_id": "mp-149", "formula": "Si", "crystal_system": "cubic", "band_gap": 0.61, "formation_energy_per_atom": -0.51, "is_stable": True},
    {"material_id": "mp-2534", "formula": "GaAs", "crystal_system": "cubic", "band_gap": 0.19, "formation_energy_per_atom": -0.21, "is_stable": True},
    {"material_id": "mp-1265", "formula": "ZnO", "crystal_system": "hexagonal", "band_gap": 0.78, "formation_energy_per_atom": -1.5, "is_stable": True},
    {"material_id": "mp-22862", "formula": "NaCl", "crystal_system": "cubic", "band_gap": 4.85, "formation_energy_per_atom": -2.0, "is_stable": True},
    {"material_id": "mp-19017", "formula": "Fe2O3", "crystal_system": "trigonal", "band_gap": 0.0, "formation_energy_per_atom": -1.6, "is_stable": True},
]

def fetch_materials_project(limit=20):
    from mp_api.client import MPRester  # requires `pip install mp-api`
    with MPRester(MP_API_KEY) as mpr:
        docs = mpr.materials.summary.search(
            num_chunks=1, chunk_size=limit,
            fields=["material_id", "formula_pretty", "symmetry", "band_gap", "formation_energy_per_atom", "is_stable"],
        )
    records = [{
        "material_id": str(d.material_id),
        "formula": d.formula_pretty,
        "crystal_system": str(d.symmetry.crystal_system) if d.symmetry else None,
        "band_gap": d.band_gap,
        "formation_energy_per_atom": d.formation_energy_per_atom,
        "is_stable": d.is_stable,
    } for d in docs]
    return pd.DataFrame(records)

if MP_API_KEY:
    try:
        df_records = fetch_materials_project()
    except Exception as e:
        print(f"Materials Project fetch failed ({e}), using fallback sample")
        df_records = pd.DataFrame(FALLBACK_RECORDS)
else:
    print("MP_API_KEY not set, using fallback sample")
    df_records = pd.DataFrame(FALLBACK_RECORDS)

df_records.to_json(f"{DATA_DIR}/structured_records.jsonl", orient="records", lines=True)
print(df_records.shape)
df_records.head()

MP_API_KEY not set, using fallback sample
(5, 6)


,material_id,formula,crystal_system,band_gap,formation_energy_per_atom,is_stable
0,mp-149,Si,cubic,0.61,-0.51,True
1,mp-2534,GaAs,cubic,0.19,-0.21,True
2,mp-1265,ZnO,hexagonal,0.78,-1.50,True
3,mp-22862,NaCl,cubic,4.85,-2.00,True
4,mp-19017,Fe2O3,trigonal,0.00,-1.60,True


## 3. SFT instruction examples

Templated question/answer pairs generated directly from the structured records.

In [62]:
def make_sft_examples(df):
    examples = []
    for _, r in df.iterrows():
        examples.append({
            "instruction": f"What is the band gap of {r['formula']}?",
            "response": f"{r['formula']} has a band gap of {r['band_gap']:.2f} eV, based on its {r['crystal_system']} crystal structure.",
        })
        examples.append({
            "instruction": f"Is {r['formula']} thermodynamically stable?",
            "response": f"{r['formula']} is {'stable' if r['is_stable'] else 'not stable'}, with a formation energy of {r['formation_energy_per_atom']:.2f} eV/atom.",
        })
    return pd.DataFrame(examples)

df_sft = make_sft_examples(df_records)
df_sft.to_json(f"{DATA_DIR}/sft_examples.jsonl", orient="records", lines=True)
print(df_sft.shape)
df_sft.head()

(10, 2)


,instruction,response
0,What is the band gap of Si?,"Si has a band gap of 0.61 eV, based on its cub..."
1,Is Si thermodynamically stable?,"Si is stable, with a formation energy of -0.51..."
2,What is the band gap of GaAs?,"GaAs has a band gap of 0.19 eV, based on its c..."
3,Is GaAs thermodynamically stable?,"GaAs is stable, with a formation energy of -0...."
4,What is the band gap of ZnO?,"ZnO has a band gap of 0.78 eV, based on its he..."


## 4. DPO preference pairs

Chosen = grounded in the record. Rejected = a plausible-sounding but unsupported/hallucinated claim.

In [63]:
def make_dpo_pairs(df):
    pairs = []
    for _, r in df.iterrows():
        prompt = f"What is the band gap of {r['formula']} and is it stable?"
        chosen = (
            f"Based on the available data, {r['formula']} has a band gap of {r['band_gap']:.2f} eV "
            f"and a formation energy of {r['formation_energy_per_atom']:.2f} eV/atom, so it is "
            f"{'considered stable' if r['is_stable'] else 'not considered stable'}."
        )
        rejected = (
            f"{r['formula']} is a well-known high-performance material with an exceptionally large "
            f"band gap, making it ideal for next-generation power electronics."
        )
        pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
    return pd.DataFrame(pairs)

df_dpo = make_dpo_pairs(df_records)
df_dpo.to_json(f"{DATA_DIR}/dpo_pairs.jsonl", orient="records", lines=True)
print(df_dpo.shape)
df_dpo.head()

(5, 3)


,prompt,chosen,rejected
0,What is the band gap of Si and is it stable?,"Based on the available data, Si has a band gap...",Si is a well-known high-performance material w...
1,What is the band gap of GaAs and is it stable?,"Based on the available data, GaAs has a band g...",GaAs is a well-known high-performance material...
2,What is the band gap of ZnO and is it stable?,"Based on the available data, ZnO has a band ga...",ZnO is a well-known high-performance material ...
3,What is the band gap of NaCl and is it stable?,"Based on the available data, NaCl has a band g...",NaCl is a well-known high-performance material...
4,What is the band gap of Fe2O3 and is it stable?,"Based on the available data, Fe2O3 has a band ...",Fe2O3 is a well-known high-performance materia...
